# Tutorial: Regularization

> **Type:** Tutorial — follow along step by step. Your only job is to make it work.  
> **Time:** ~1.5 hours  
> **Prerequisites:** Modules 13 (cross-validation), 16 (model evaluation)  
> **What you'll build:** A visual demonstration of overfitting + all three regularisation methods (Ridge, Lasso, Elastic Net) applied to real data, with regularisation paths showing how the penalty reshapes the model.

---

## What We're Building

We'll first *create* overfitting on purpose so you can see exactly what it looks like. Then we'll fix it with L1, L2, and Elastic Net regularisation — and visualise the regularisation paths to understand how each method differently constrains the model.

## Step 0: Setup

In [ ]:
# !pip install scikit-learn matplotlib numpy pandas

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import (
    LinearRegression, Ridge, Lasso, ElasticNet,
    RidgeCV, LassoCV, ElasticNetCV
)
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import mean_squared_error, r2_score

SEED = 42
np.random.seed(SEED)
plt.rcParams['figure.figsize'] = (12, 5)
print('Setup complete.')

---
## Part 1: Creating and Visualising Overfitting

### Step 1: Generate a Simple 1D Dataset

In [ ]:
# True function: y = sin(x) + noise
n_samples = 30
X_1d = np.sort(np.random.uniform(-3, 3, n_samples))
y_1d = np.sin(X_1d) + np.random.normal(0, 0.3, n_samples)
X_1d_test = np.linspace(-3, 3, 200)

plt.figure(figsize=(8, 4))
plt.scatter(X_1d, y_1d, color='#4E79A7', s=50, zorder=3, label='Training data')
plt.plot(X_1d_test, np.sin(X_1d_test), 'k--', linewidth=1.5, label='True function sin(x)')
plt.xlabel('x'); plt.ylabel('y')
plt.title('The Dataset (30 noisy samples of sin(x))', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

### Step 2: Fit Polynomials of Increasing Degree

In [ ]:
degrees = [1, 5, 15]
colours = ['#F28E2B', '#59A14F', '#E15759']
X_col   = X_1d.reshape(-1, 1)

train_errors, test_errors = [], []

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, deg, colour in zip(axes, degrees, colours):
    pipe = make_pipeline(PolynomialFeatures(deg), LinearRegression())
    pipe.fit(X_col, y_1d)
    
    X_test_col = X_1d_test.reshape(-1, 1)
    y_fit  = pipe.predict(X_test_col)
    
    # Train error
    train_pred  = pipe.predict(X_col)
    train_rmse  = np.sqrt(mean_squared_error(y_1d, train_pred))
    train_errors.append(train_rmse)
    
    # Approximate test error via CV
    cv_rmse = np.sqrt(-cross_val_score(pipe, X_col, y_1d, cv=5,
                                        scoring='neg_mean_squared_error').mean())
    test_errors.append(cv_rmse)
    
    ax.scatter(X_1d, y_1d, color='#4E79A7', s=30, zorder=3, label='Data')
    ax.plot(X_1d_test, np.sin(X_1d_test), 'k--', linewidth=1, label='True fn')
    ax.plot(X_1d_test, np.clip(y_fit, -3, 3), color=colour, linewidth=2, label=f'Degree {deg}')
    ax.set_title(f'Degree {deg}\nTrain RMSE={train_rmse:.3f}  CV RMSE={cv_rmse:.3f}',
                 fontweight='bold')
    ax.set_ylim(-2.5, 2.5)
    ax.legend(fontsize=8)

plt.suptitle('Underfitting → Good Fit → Overfitting', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('polynomial_overfitting.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nSummary:")
for deg, tr, te in zip(degrees, train_errors, test_errors):
    print(f"  Degree {deg:2d}: Train RMSE={tr:.3f}  CV RMSE={te:.3f}")

> **What just happened?** Degree 1 underfits — too simple. Degree 5 fits well. Degree 15 has near-zero training error but terrible CV error — it memorised the noise. The gap between train and CV error is the signature of overfitting.

---
## Part 2: Ridge Regression (L2)

### Step 3: Fix Overfitting with Ridge

In [ ]:
alphas = [0.001, 0.1, 1.0, 10.0, 100.0]

fig, axes = plt.subplots(1, len(alphas), figsize=(18, 4), sharey=True)

for ax, alpha in zip(axes, alphas):
    pipe = make_pipeline(PolynomialFeatures(15), StandardScaler(), Ridge(alpha=alpha))
    pipe.fit(X_col, y_1d)
    y_fit = pipe.predict(X_1d_test.reshape(-1, 1))
    cv_rmse = np.sqrt(-cross_val_score(pipe, X_col, y_1d, cv=5,
                                        scoring='neg_mean_squared_error').mean())
    ax.scatter(X_1d, y_1d, color='#4E79A7', s=20, zorder=3)
    ax.plot(X_1d_test, np.sin(X_1d_test), 'k--', linewidth=1)
    ax.plot(X_1d_test, np.clip(y_fit, -2.5, 2.5), color='#E15759', linewidth=2)
    ax.set_title(f'α={alpha}\nCV RMSE={cv_rmse:.3f}', fontweight='bold')
    ax.set_ylim(-2.5, 2.5)

plt.suptitle('Ridge (L2) on Degree-15 Polynomial — Increasing α', fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('ridge_regularisation.png', dpi=150, bbox_inches='tight')
plt.show()

### Step 4: Regularisation Path for Ridge

In [ ]:
from sklearn.linear_model import ridge_regression

# Transform features
poly = PolynomialFeatures(15)
scaler = StandardScaler()
X_poly = scaler.fit_transform(poly.fit_transform(X_col))

alphas_path = np.logspace(-3, 4, 100)
coef_paths = []

for alpha in alphas_path:
    ridge = Ridge(alpha=alpha, fit_intercept=False)
    ridge.fit(X_poly, y_1d)
    coef_paths.append(ridge.coef_)

coef_paths = np.array(coef_paths)

plt.figure(figsize=(10, 5))
for i in range(coef_paths.shape[1]):
    plt.plot(np.log10(alphas_path), coef_paths[:, i], linewidth=1, alpha=0.6)
plt.xlabel('log₁₀(α)  →  increasing regularisation')
plt.ylabel('Coefficient value')
plt.title('Ridge Regularisation Path\n(all 16 polynomial coefficients)', fontweight='bold')
plt.axvline(0, color='grey', linestyle='--', linewidth=1)
plt.tight_layout()
plt.savefig('ridge_path.png', dpi=150, bbox_inches='tight')
plt.show()

print("As α → ∞, ALL coefficients shrink toward zero.")
print("Ridge keeps all features but makes them small.")

---
## Part 3: Lasso Regression (L1)

### Step 5: Lasso Drives Coefficients to Exactly Zero

In [ ]:
alphas_lasso = np.logspace(-4, 0, 100)
lasso_coef_paths = []

for alpha in alphas_lasso:
    lasso = Lasso(alpha=alpha, max_iter=10000, fit_intercept=False)
    lasso.fit(X_poly, y_1d)
    lasso_coef_paths.append(lasso.coef_)

lasso_coef_paths = np.array(lasso_coef_paths)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Lasso path
for i in range(lasso_coef_paths.shape[1]):
    axes[0].plot(np.log10(alphas_lasso), lasso_coef_paths[:, i], linewidth=1, alpha=0.7)
axes[0].set_xlabel('log₁₀(α)  →  increasing regularisation')
axes[0].set_ylabel('Coefficient value')
axes[0].set_title('Lasso (L1) Regularisation Path', fontweight='bold')

# Count non-zero coefficients
n_nonzero = (np.abs(lasso_coef_paths) > 1e-4).sum(axis=1)
axes[1].plot(np.log10(alphas_lasso), n_nonzero, color='#E15759', linewidth=2)
axes[1].set_xlabel('log₁₀(α)')
axes[1].set_ylabel('Number of non-zero coefficients')
axes[1].set_title('Lasso Feature Selection Effect\n(features driven to zero)', fontweight='bold')

plt.tight_layout()
plt.savefig('lasso_path.png', dpi=150, bbox_inches='tight')
plt.show()

print("Lasso drives coefficients to EXACTLY zero — this is feature selection.")
print("Ridge shrinks them but never zeros them out.")

---
## Part 4: Apply to Real Data and Compare All Three

### Step 6: Breast Cancer Dataset — Logistic with L1/L2/ElasticNet

In [ ]:
# Use breast cancer dataset — 30 features
bc = load_breast_cancer()
X_bc, y_bc = bc.data, bc.target
feature_names = bc.feature_names

scaler_bc = StandardScaler()
X_bc_s = scaler_bc.fit_transform(X_bc)

# Cross-validated selection of best alpha
ridge_cv    = RidgeCV(alphas=np.logspace(-3, 3, 100), cv=5)
lasso_cv    = LassoCV(alphas=np.logspace(-4, 1, 100), cv=5, max_iter=10000)
enet_cv     = ElasticNetCV(alphas=np.logspace(-4, 1, 50), l1_ratio=[0.1, 0.5, 0.9], cv=5)

# Note: RidgeCV is a regressor, so we use it on a regression-like task
# For fair comparison, use cross_val_score on classification metric
from sklearn.linear_model import LogisticRegressionCV

results = {}
for penalty, solver in [('l2', 'lbfgs'), ('l1', 'saga'), ('elasticnet', 'saga')]:
    kwargs = {'penalty': penalty, 'solver': solver, 'max_iter': 10000,
              'Cs': 20, 'cv': 5, 'random_state': SEED}
    if penalty == 'elasticnet':
        kwargs['l1_ratios'] = [0.1, 0.5, 0.9]
    model = LogisticRegressionCV(**kwargs)
    model.fit(X_bc_s, y_bc)
    scores = cross_val_score(model, X_bc_s, y_bc, cv=5, scoring='roc_auc')
    n_nonzero = (np.abs(model.coef_[0]) > 1e-4).sum()
    results[penalty] = {
        'ROC-AUC': scores.mean(),
        'std': scores.std(),
        'features_used': n_nonzero,
        'best_C': model.C_[0],
    }

print(f"{'Penalty':15s} {'ROC-AUC':>10} {'±':>6} {'Features used':>15} {'Best C':>10}")
print("-" * 60)
for penalty, r in results.items():
    print(f"{penalty:15s} {r['ROC-AUC']:>10.4f} {r['std']:>6.4f} {r['features_used']:>15} {r['best_C']:>10.4f}")

### Step 7: Visualise Which Features Lasso Zeroes Out

In [ ]:
# Refit Lasso logistic regression and inspect coefficients
from sklearn.linear_model import LogisticRegression

# Use best C from lasso_cv above
lasso_model = LogisticRegressionCV(
    penalty='l1', solver='saga', Cs=30, cv=5, max_iter=10000, random_state=SEED
)
lasso_model.fit(X_bc_s, y_bc)
coefs = lasso_model.coef_[0]

# Sort by absolute value
idx   = np.argsort(np.abs(coefs))[::-1]
coefs_sorted  = coefs[idx]
names_sorted  = feature_names[idx]
colours = ['#E15759' if c != 0 else '#CCCCCC' for c in coefs_sorted]

plt.figure(figsize=(14, 5))
plt.bar(range(len(coefs_sorted)), coefs_sorted, color=colours)
plt.xticks(range(len(names_sorted)), names_sorted, rotation=90, fontsize=8)
plt.axhline(0, color='black', linewidth=1)
plt.title(f'Lasso Coefficients (grey = zeroed out, {(coefs==0).sum()} features eliminated)',
          fontweight='bold')
plt.ylabel('Coefficient value')
plt.tight_layout()
plt.savefig('lasso_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()

> **What just happened?** The grey bars are features Lasso eliminated — set their coefficient exactly to zero. Ridge would have kept all features with small non-zero values. This is the fundamental difference: Lasso selects; Ridge shrinks.

---
## What You Built

A complete visual tour of regularisation: you created overfitting from scratch with polynomial regression, fixed it with Ridge (seeing how all coefficients shrink), compared it to Lasso (seeing features eliminated entirely), and applied all three methods to real classification data with cross-validated alpha selection.

## What's Next

- Read `explanation.md` for the bias-variance trade-off and the geometry of L1 vs L2.
- Use `howto.md` when you need to apply regularisation to your own models.
- Keep `reference.md` open for the method comparison table.
- **Extension challenge:** Use `ElasticNetCV` to find the best `l1_ratio` on the breast cancer data. Try values from 0.1 to 0.9. What l1_ratio gives the best CV score? Does it keep more or fewer features than pure L1?

---

**Next module:** `19_unsupervised_learning` — shifting from labelled to unlabelled data: clustering and anomaly detection.